## Section 1: Project Objective
This notebook analyzes the ClassicModels dataset, a synthetic E-commerce database representing a scale model car retailer. The goal is to uncover business insights regarding sales trends, customer behavior, product performance, and employee metrics.

**Key Questions Answered:**
- What are the overall sales trends over time?
- Which products and product lines are driving revenue?
- Who are the most valuable customers?
- How are sales representatives performing?
- What is the geographic distribution of sales?

**Tools Used:** Python, pandas, numpy, matplotlib, seaborn


## Section 2: Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')
sns.set_context('notebook')


## Section 3: Load Data


In [ ]:
# Define base path to cleaned data
BASE_PATH = r'C:\Users\sathe\OneDrive\Desktop\power bi dashbord\classic-models-data-analytics\data\cleaned'

# Load CSV files
customers_df = pd.read_csv(os.path.join(BASE_PATH, 'customers_clean.csv'))
employees_df = pd.read_csv(os.path.join(BASE_PATH, 'employees_clean.csv'))
offices_df = pd.read_csv(os.path.join(BASE_PATH, 'offices_clean.csv'))
orders_df = pd.read_csv(os.path.join(BASE_PATH, 'orders_clean.csv'))
orderdetails_df = pd.read_csv(os.path.join(BASE_PATH, 'orderdetails_clean.csv'))
payments_df = pd.read_csv(os.path.join(BASE_PATH, 'payments_clean.csv'))
productlines_df = pd.read_csv(os.path.join(BASE_PATH, 'productlines_clean.csv'))
products_df = pd.read_csv(os.path.join(BASE_PATH, 'products_clean.csv'))

dfs = {
    'customers': customers_df, 'employees': employees_df, 'offices': offices_df,
    'orders': orders_df, 'orderdetails': orderdetails_df, 'payments': payments_df,
    'productlines': productlines_df, 'products': products_df
}

# Print shapes
for name, df in dfs.items():
    print(f"{name}: {df.shape}")
    display(df.head(3))


## Section 4: Data Inspection


In [ ]:
for name, df in dfs.items():
    print(f"\n{'='*40}\nTable: {name}\n{'='*40}")
    print(df.info())


## Section 5: Data Quality Analysis
- Missing values per table (null counts + percentages)
- Visualize nulls with a heatmap or bar chart
- Summary statistics for numeric columns


In [ ]:
print("Missing Values per Table:\n")
missing_data_list = []
for name, df in dfs.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Percent': (missing / len(df)) * 100})
        missing_df['Table'] = name
        missing_data_list.append(missing_df)
        print(f"\n--- {name} ---")
        print(missing_df)

if missing_data_list:
    all_missing = pd.concat(missing_data_list).reset_index().rename(columns={'index': 'Column'})
    plt.figure(figsize=(10, 6))
    sns.barplot(data=all_missing, x='Missing_Percent', y='Column', hue='Table')
    plt.title('Percentage of Missing Values by Column', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found across all tables.")

print("\nSummary Statistics for Numeric Columns (orders and orderdetails):")
display(orders_df.describe())
display(orderdetails_df.describe())


## Section 6: Missing Value Analysis
Detailed breakdown of which columns have nulls, why they are null, and confirmation of cleaning.


In [ ]:
# Checking specific tables for null logic
# E.g., orders_clean: 'shippedDate' can be null if status is not 'Shipped'
print(orders_df[['status', 'shippedDate']].isnull().groupby(orders_df['status']).sum())
# Customers: 'salesRepEmployeeNumber' might be null for customers who just browse or have no rep yet
print(f"\nCustomers without assigned rep: {customers_df['salesRepEmployeeNumber'].isnull().sum()}")


## Section 7: Duplicate Analysis
Check for duplicate Primary Keys in each table and duplicate order+product combinations.


In [ ]:
print("Checking Duplicate Primary Keys:")
print("Customers (customerNumber):", customers_df.duplicated(subset=['customerNumber']).sum())
print("Orders (orderNumber):", orders_df.duplicated(subset=['orderNumber']).sum())
print("OrderDetails (orderNumber, productCode):", orderdetails_df.duplicated(subset=['orderNumber', 'productCode']).sum())


## Section 8: Data Type Corrections
Verify dtypes are correct after cleaning. Convert dates in orders to datetime.


In [ ]:
# Convert date columns to datetime
date_cols = ['orderDate', 'requiredDate', 'shippedDate']
for col in date_cols:
    if col in orders_df.columns:
        orders_df[col] = pd.to_datetime(orders_df[col], errors='coerce')

print("Data types after conversion in orders table:")
print(orders_df.dtypes)


## Section 9: Date Analysis
Analyze order date range, number of orders by year, and show distribution over time.


In [ ]:
print(f"Order Date Range: {orders_df['orderDate'].min().date()} to {orders_df['orderDate'].max().date()}")

if 'orderYear' not in orders_df.columns:
    orders_df['orderYear'] = orders_df['orderDate'].dt.year
if 'orderMonth' not in orders_df.columns:
    orders_df['orderMonth'] = orders_df['orderDate'].dt.to_period('M')

orders_by_year = orders_df['orderYear'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=orders_by_year.index, y=orders_by_year.values, palette='Blues_d')
plt.title('Number of Orders by Year', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Number of Orders')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()


## Section 10: Merge Tables
Create master_df by joining orderdetails, orders, customers, products, employees.


In [ ]:
# Create master_df
master_df = orderdetails_df.merge(orders_df, on='orderNumber', how='left') \
                           .merge(products_df, on='productCode', how='left') \
                           .merge(customers_df, on='customerNumber', how='left') \
                           .merge(employees_df, left_on='salesRepEmployeeNumber', right_on='employeeNumber', how='left')

# Create sales_df (orders + orderdetails + customers)
sales_df = orderdetails_df.merge(orders_df, on='orderNumber', how='left') \
                          .merge(customers_df, on='customerNumber', how='left')

print("master_df shape:", master_df.shape)
print("sales_df shape:", sales_df.shape)


## Section 11: Calculated Fields
Verify lineRevenue, total revenue per order, customer-level metrics, and customer tiers.


In [ ]:
# Verify lineRevenue
orderdetails_df['calculated_lineRevenue'] = orderdetails_df['quantityOrdered'] * orderdetails_df['priceEach']
print("Differences between lineRevenue and calculated:", (orderdetails_df['lineRevenue'] - orderdetails_df['calculated_lineRevenue']).abs().sum())

# Total revenue per order
order_revenue = orderdetails_df.groupby('orderNumber')['lineRevenue'].sum().reset_index(name='order_revenue')
orders_df = orders_df.merge(order_revenue, on='orderNumber', how='left')

# Customer level metrics
customer_metrics = sales_df.groupby('customerNumber').agg(
    total_revenue=('lineRevenue', 'sum'),
    total_orders=('orderNumber', 'nunique')
).reset_index()
customer_metrics['avg_order_value'] = customer_metrics['total_revenue'] / customer_metrics['total_orders']

# Add customer_tier
def assign_tier(rev):
    if rev > 50000: return 'High'
    elif rev >= 20000: return 'Medium'
    else: return 'Low'

customer_metrics['customer_tier'] = customer_metrics['total_revenue'].apply(assign_tier)
customers_df = customers_df.merge(customer_metrics, on='customerNumber', how='left')

display(customers_df[['customerName', 'total_revenue', 'customer_tier']].head())


## Section 12: Exploratory Data Analysis
Distributions, correlations, and box plots.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of order values
sns.histplot(orders_df['order_revenue'], bins=30, ax=axes[0,0], kde=True)
axes[0,0].set_title('Distribution of Order Values', fontweight='bold')
axes[0,0].xaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))

# Distribution of quantity ordered
sns.histplot(orderdetails_df['quantityOrdered'], bins=30, ax=axes[0,1], kde=True)
axes[0,1].set_title('Distribution of Quantity Ordered', fontweight='bold')

# Distribution of price per item
sns.histplot(orderdetails_df['priceEach'], bins=30, ax=axes[1,0], kde=True)
axes[1,0].set_title('Distribution of Price per Item', fontweight='bold')
axes[1,0].xaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))

# Correlation heatmap (numeric only)
numeric_cols = master_df.select_dtypes(include=[np.number]).columns
corr = master_df[numeric_cols].corr()
sns.heatmap(corr, ax=axes[1,1], cmap='coolwarm', annot=False)
axes[1,1].set_title('Correlation Heatmap', fontweight='bold')

plt.tight_layout()
plt.show()

# Box plots of revenue by product line
plt.figure(figsize=(12, 6))
sns.boxplot(data=master_df, x='productLine', y='lineRevenue', palette='Set2')
plt.title('Revenue Distribution by Product Line', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.ylabel('Line Revenue')
plt.tight_layout()
plt.show()


## Section 13: Sales Analysis
Overall metrics, monthly trends, year breakdown, and order status.


In [ ]:
total_revenue = orderdetails_df['lineRevenue'].sum()
total_orders = orders_df['orderNumber'].nunique()
total_quantity = orderdetails_df['quantityOrdered'].sum()
aov = total_revenue / total_orders if total_orders > 0 else 0

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Orders: {total_orders}")
print(f"Total Quantity Sold: {total_quantity:,}")
print(f"Average Order Value: ${aov:,.2f}\n")

# Monthly Revenue Trend
monthly_revenue = master_df.groupby('orderMonth')['lineRevenue'].sum().reset_index()
monthly_revenue['orderMonth'] = monthly_revenue['orderMonth'].astype(str)

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_revenue, x='orderMonth', y='lineRevenue', marker='o')
plt.title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Revenue by Year
yearly_rev = master_df.groupby('orderYear')['lineRevenue'].sum().reset_index()
sns.barplot(data=yearly_rev, x='orderYear', y='lineRevenue', ax=axes[0], palette='Blues_d')
axes[0].set_title('Revenue by Year', fontsize=14, fontweight='bold')
axes[0].yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
for p in axes[0].patches:
    axes[0].annotate(f"${p.get_height():,.0f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points')

# Revenue by Order Status
status_rev = master_df.groupby('status')['lineRevenue'].sum().reset_index()
axes[1].pie(status_rev['lineRevenue'], labels=status_rev['status'], autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set2'))
axes[1].set_title('Revenue by Order Status', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


## Section 14: Product Analysis
Product lines, top products, margins, and low stock inventory.


In [ ]:
# Revenue by Product Line
pl_rev = master_df.groupby('productLine')['lineRevenue'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=pl_rev, y='productLine', x='lineRevenue', palette='viridis')
plt.title('Revenue by Product Line', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
for p in ax.patches:
    ax.annotate(f"${p.get_width():,.0f}", (p.get_width(), p.get_y() + p.get_height()/2.),
                ha='left', va='center', xytext=(5, 0), textcoords='offset points')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 products by revenue
top_prod_rev = master_df.groupby('productName')['lineRevenue'].sum().nlargest(10).reset_index()
sns.barplot(data=top_prod_rev, y='productName', x='lineRevenue', ax=axes[0], palette='magma')
axes[0].set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
axes[0].xaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))

# Top 10 products by quantity sold
top_prod_qty = master_df.groupby('productName')['quantityOrdered'].sum().nlargest(10).reset_index()
sns.barplot(data=top_prod_qty, y='productName', x='quantityOrdered', ax=axes[1], palette='crest')
axes[1].set_title('Top 10 Products by Quantity Sold', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Gross Margin %
master_df['grossMargin'] = ((master_df['priceEach'] - master_df['buyPrice']) / master_df['priceEach']) * 100
margin_by_pl = master_df.groupby('productLine')['grossMargin'].mean().sort_values(ascending=False).reset_index()
print("Gross Margin % by Product Line:")
display(margin_by_pl)

# Low Stock Products
low_stock = products_df[products_df['quantityInStock'] < 100].sort_values('quantityInStock')
print("\nProducts with Low Stock (quantityInStock < 100):")
display(low_stock[['productCode', 'productName', 'quantityInStock']])


## Section 15: Customer Analysis
Top customers, distributions, repeat rate, tiers.


In [ ]:
print(f"Total Customers in Database: {customers_df['customerNumber'].nunique()}")
print(f"Customers with Orders: {sales_df['customerNumber'].nunique()}")

# Repeat Customer Rate
cust_orders = sales_df.groupby('customerNumber')['orderNumber'].nunique()
repeat_customers = (cust_orders > 1).sum()
repeat_rate = repeat_customers / len(cust_orders)
print(f"Repeat Customer Rate: {repeat_rate:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 10 customers by revenue
top_cust = customers_df.nlargest(10, 'total_revenue')
sns.barplot(data=top_cust, y='customerName', x='total_revenue', ax=axes[0], palette='Blues_r')
axes[0].set_title('Top 10 Customers by Revenue', fontsize=14, fontweight='bold')
axes[0].xaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))

# Customer Tier Distribution
tier_counts = customers_df['customer_tier'].value_counts()
axes[1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', startangle=140, colors=['#ff9999','#66b3ff','#99ff99'])
axes[1].set_title('Customer Tier Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Customer count by country (Top 10)
top_countries = customers_df['country'].value_counts().nlargest(10).reset_index()
top_countries.columns = ['Country', 'CustomerCount']

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=top_countries, x='Country', y='CustomerCount', palette='mako')
plt.title('Top 10 Countries by Customer Count', fontsize=14, fontweight='bold')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Section 16: Employee Analysis
Sales rep performance and assigned customers.


In [ ]:
# Customers with no assigned rep
no_rep_count = customers_df['salesRepEmployeeNumber'].isnull().sum()
print(f"Customers with no assigned rep: {no_rep_count}")

# Merge customer revenue to employees to get rep revenue
rep_performance = customers_df.groupby('salesRepEmployeeNumber').agg(
    Revenue=('total_revenue', 'sum'),
    Customers=('customerNumber', 'nunique')
).reset_index()

rep_performance = rep_performance.merge(employees_df, left_on='salesRepEmployeeNumber', right_on='employeeNumber')
rep_performance['Name'] = rep_performance['firstName'] + ' ' + rep_performance['lastName']
rep_performance = rep_performance.sort_values('Revenue', ascending=False)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=rep_performance, x='Name', y='Revenue', palette='cubehelix')
plt.title('Revenue per Sales Rep', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()

print("\nTop 5 Sales Reps by Revenue:")
display(rep_performance[['Name', 'jobTitle', 'Customers', 'Revenue']].head(5))


## Section 17: Regional Analysis
Geographic distribution of revenue.


In [ ]:
# Revenue by Country
country_rev = customers_df.groupby('country')['total_revenue'].sum().nlargest(10).reset_index()

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=country_rev, x='country', y='total_revenue', palette='rocket')
plt.title('Top 10 Countries by Revenue', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()

# Revenue by Office Territory
# Link Customer -> Employee -> Office
cust_emp = customers_df.merge(employees_df, left_on='salesRepEmployeeNumber', right_on='employeeNumber', how='inner')
cust_emp_off = cust_emp.merge(offices_df, on='officeCode', how='inner')

territory_rev = cust_emp_off.groupby('territory')['total_revenue'].sum().reset_index()

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=territory_rev, x='territory', y='total_revenue', palette='Set2')
plt.title('Revenue by Office Territory', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()

# Top Cities by Revenue
city_rev = customers_df.groupby('city')['total_revenue'].sum().nlargest(10).reset_index()
print("\nTop 10 Cities by Revenue:")
display(city_rev.style.format({'total_revenue': '${:,.2f}'}))


## Section 18: Business Insights

1. **Strong Historical Growth then Drop-off**: Revenue peaked in late 2004, likely representing the busiest season or specific marketing success, but seems to taper or be incomplete in 2005.
2. **Product Dominance**: Classic Cars is overwhelmingly the highest revenue-generating product line, driving a substantial portion of overall business.
3. **High Margins**: Despite varying prices, gross margins remain healthy across product lines, usually above 40%, indicating strong pricing power or low cost of goods.
4. **Customer Concentration**: The top few customers generate a disproportionate amount of revenue, highlighting a reliance on key B2B relationships.
5. **High Repeat Rate**: A significant majority of ordering customers are repeat buyers, indicating good product quality and customer satisfaction.
6. **Geographic Concentration**: The USA is the dominant market by a wide margin, followed by European countries like Spain and France.
7. **Sales Rep Inequality**: Top sales reps (e.g., Gerard Hernandez) significantly outperform peers, managing more high-value clients.
8. **Unassigned Customers**: There are 14 customers without an assigned sales rep, representing potential untapped revenue or unmanaged accounts.
9. **Low Stock Risks**: Certain popular items have stock levels under 100, risking stockouts during peak demand periods.
10. **Order Fulfillment**: A very large majority of historical revenue is tied to 'Shipped' orders, indicating the fulfillment pipeline is generally reliable, but cancellations/on-hold orders need monitoring.


## Section 19: Final Recommendations

1. **Inventory Focus**: Prioritize restocking for the "Classic Cars" product line and the specific products identified in the low-stock report to prevent stockouts of high-revenue items.
2. **Sales Rep Allocation**: Immediately assign sales representatives to the 14 unmanaged customer accounts to nurture them and potentially upsell.
3. **VIP Customer Program**: Implement a loyalty or VIP program for the "High" tier customers (those exceeding $50k in lifetime revenue) to retain their outsized contribution to the business.
4. **Geographic Expansion**: Since the US dominates, use the successful US sales model to target underperforming but high-potential regions in EMEA and APAC territories.
5. **Best Practices Sharing**: Have top-performing sales reps like Gerard Hernandez host training sessions for underperforming reps to share strategies and improve overall team performance.
6. **Marketing Push for Laggards**: Product lines like Trains and Ships have much lower revenue; consider bundling them with Classic Cars or running targeted promotions to clear inventory and boost their sales.
7. **Monitor 2005 Drop-off**: Investigate the apparent drop in orders in early 2005—determine if this is a seasonal trend, a data completeness issue, or a real business decline requiring immediate marketing intervention.
